# IV-8 音楽信号と非線形システム（ノートブック版）

教科書の IV-8 の主な演習を 1 つにまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。計算は全部で 1 分ほどです（Mac で約 35 秒）。5 節がいちばん長く、十数秒〜数十秒かかります）。

| 節 | 試すこと | 教科書 | 計算の時間の目安 |
|---|---|---|---|
| 1 | 楽音：倍音を足していく（スペクトルの重心） | IV-8-3-1、手順 A | 数秒 |
| 2 | ピッチ推定：自前の YIN と librosa.yin・pyin | IV-8-3-3・3-4、手順 B | 数秒〜十数秒 |
| 3 | 音楽の NMF（調波構造の制約つき）と、しきい値の選び方 | IV-8-3-5、手順 C | 十数秒 |
| 4 | HPSS：自前のメディアンフィルタと librosa | IV-8-3-6、手順 D | 数秒 |
| 5 | 被り音の抑圧：時間チャネル NMF（短い版） | IV-8-3-7、手順 E | 数十秒 |
| 6 | ひずみ：THD と相互変調 | IV-8-3-9、手順 F | 数秒 |
| 7 | ダイオードクリッパ：静的な解と数値解法 | IV-8-3-10・3-11、手順 G | 数秒 |
| 8 | 折り返しとオーバーサンプリング | IV-8-3-12、手順 H | 数秒 |

- 各節の後のセルで、教科書の手順の「確かめること」の主な数値を `ok()` で確かめます（合わないと「★確かめ失敗」で止まります。短い版の節は、短くした分の幅を持たせてあります）。
- 2 節で、librosa の例の音源 `trumpet`（`librosa.ex("trumpet")`）を**初回だけダウンロード**します（数百 KB。保存先は librosa の既定のキャッシュのフォルダ）。ダウンロードできないとき（ネットにつながらないなど）は、自動で合成のトランペット風の音に切り替えます。その場合、trumpet の行の数値は教科書と違い、pYIN との一致の割合の確かめは省きます。

- 最初のセルが、共通ライブラリ（`common/stft.py`・`common/music.py`・`common/nonlinear.py`、教科書の IV-2・IV-8 の 6 節と同じ内容）と `nmf8.py` を、このノートブックの隣に書き出します。
- **音を聞くときの注意**：再生ボタンは `listen()` が作ります。中で最大値を 0.3（ひずんだ音は実効値を −30 dBFS）にそろえ、`IPython.display.Audio(..., normalize=False)` で鳴らします。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前に音量を小さくしてください。
- 計算時間を短くするため、被り音は論文の値の 1 設定・混合 5 個、クリッパは短い音にしてあります。**数値は教科書の表と少し違います**。教科書の数値はスクリプト版（`~/home-lab/partIV/iv8/`）で確かめてください。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 足りないパッケージだけ入れる（Colab には numpy・scipy・librosa・matplotlib・scikit-learn が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("fast_bss_eval", "fast_bss_eval"), ("librosa", "librosa")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "librosa", librosa.__version__)
import os
os.makedirs("common", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""


In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out


In [ ]:
%%writefile common/music.py
"""第IV部の共通ライブラリ：楽音・音名・ピッチ推定（IV-8 で定義）

  音名と周波数（平均律、A4 = 440 Hz、C4 = MIDI 60）
    f = midi_to_hz(69)              → 440.0
    m = hz_to_midi(261.63)          → 60.0 くらい
    s = note_name(61)               → "C#4"
    c = cents(f, f_ref)             → 1200·log2(f/f_ref)
  楽音を作る（どれも振幅の最大を 1 前後にした float の配列を返す）
    x = harmonic_tone(f0, fs, sec, amps=..., B=0.0)   倍音を足した音（fs/2 を超える倍音は足さない）
    e = adsr(n, fs, a, d, s, r)                       ADSR の包絡（直線）
    x = drum("kick" | "snare" | "hihat", fs, sec, seed)
  ピッチ推定
    d = yin_difference(frame, W, tau_max)             差分関数（論文の式 (6)、窓 W は固定）
    dn = cmndf(d)                                     累積平均正規化差分関数（式 (8)）
    f0, ap = yin(x, fs, fmin, fmax, frame_length=2048, hop_length=None, threshold=0.1, mode="librosa")
             mode="librosa"：librosa.yin と同じ計算（数値が一致する）
             mode="paper"  ：論文どおり窓 W を固定し、放物線補間の位置は d（正規化前）で求める
    f0 = acf_pitch(x, fs, fmin, fmax, frame_length, hop_length)   自己相関法（比較用）
    r = gross_error(f_est, f_true, tol=0.2)          相対誤差が tol を超えたら True
  クロマ
    C = chroma_simple(X, fs, L, fmin=55.0)            STFT のパワーを 12 の音名に足し合わせる
"""
import numpy as np

__all__ = ["NOTE_NAMES", "midi_to_hz", "hz_to_midi", "note_name", "cents",
           "harmonic_tone", "adsr", "drum", "frame_signal",
           "yin_difference", "cmndf", "yin", "acf_pitch", "gross_error", "chroma_simple"]

NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


# ---------------------------------------------------------------- 音名と周波数
def midi_to_hz(m, a4=440.0):
    """平均律：f = a4·2^((m − 69)/12)（MIDI 69 が A4）"""
    return a4 * 2.0 ** ((np.asarray(m, dtype=float) - 69.0) / 12.0)


def hz_to_midi(f, a4=440.0):
    """m = 69 + 12·log2(f/a4)（小数のまま返す。四捨五入すると最も近い鍵盤）"""
    return 69.0 + 12.0 * np.log2(np.asarray(f, dtype=float) / a4)


def note_name(m):
    """MIDI 番号 → "A4" のような名前（C4 = 60。オクターブの番号は C で変わる）"""
    m = int(round(float(m)))
    return f"{NOTE_NAMES[m % 12]}{m // 12 - 1}"


def cents(f, f_ref):
    """f が f_ref より何セント高いか（半音 = 100 セント、1 オクターブ = 1200 セント）"""
    return 1200.0 * np.log2(np.asarray(f, dtype=float) / np.asarray(f_ref, dtype=float))


# ---------------------------------------------------------------- 楽音を作る
def adsr(n, fs, a=0.01, d=0.1, s=0.6, r=0.2):
    """ADSR の包絡（長さ n）。a・d・r は秒、s は持続のレベル（0〜1）。
    0 → 1 まで a 秒で上がり、d 秒で s まで下がり、最後の r 秒で 0 まで下がる（すべて直線）"""
    na, nd, nr = int(a * fs), int(d * fs), int(r * fs)
    ns = n - na - nd - nr
    if ns < 0:
        raise ValueError("a + d + r が音の長さより長い")
    return np.concatenate([np.linspace(0, 1, na, endpoint=False),
                           np.linspace(1, s, nd, endpoint=False),
                           np.full(ns, s),
                           np.linspace(s, 0, nr)])


def harmonic_tone(f0, fs, sec, amps="saw", n_harm=None, B=0.0, decay=None, phase=None, seed=0):
    """倍音を足した音 x(t) = Σ_k a_k sin(2π f_k t + φ_k)、f_k = k·f0·√(1 + B k²)。
      amps   : "saw"（a_k = 1/k、のこぎり波）、"square"（奇数倍音だけ 1/k）、"flat"（すべて 1）、
               または長さ K の配列（a_1, …, a_K。0 を入れると、その倍音は無い）
      B      : 弦の非調和性の係数（ピアノの弦などで正。0 なら厳密に整数倍）
      decay  : None なら一定。数（秒）を与えると、k 番目の倍音が exp(−t·k/decay) で減衰する
               （高い倍音ほど早く消える。はじいた弦・打った弦に近い）
      phase  : None なら φ_k = 0、"random" なら乱数（seed）
    fs/2 以上の倍音は足さない（折り返しを作らない）。最大値が 1 になるようにそろえて返す。"""
    n = int(round(sec * fs))
    t = np.arange(n) / fs
    if isinstance(amps, str):
        K = n_harm or int(np.floor((fs / 2) / f0))
        k = np.arange(1, K + 1)
        if amps == "saw":
            a = 1.0 / k
        elif amps == "square":
            a = np.where(k % 2 == 1, 1.0 / k, 0.0)
        elif amps == "flat":
            a = np.ones(K)
        else:
            raise ValueError('amps は "saw"・"square"・"flat" か配列')
    else:
        a = np.asarray(amps, dtype=float)
        k = np.arange(1, len(a) + 1)
    fk = k * f0 * np.sqrt(1.0 + B * k ** 2)
    rng = np.random.default_rng(seed)
    ph = rng.uniform(0, 2 * np.pi, len(k)) if phase == "random" else np.zeros(len(k))
    x = np.zeros(n)
    for ak, f, p, kk in zip(a, fk, ph, k):
        if ak == 0 or f >= fs / 2:
            continue
        env = 1.0 if decay is None else np.exp(-t * kk / decay)
        x += ak * env * np.sin(2 * np.pi * f * t + p)
    return x / (np.max(np.abs(x)) + 1e-12)


def drum(kind, fs, sec=0.5, seed=0):
    """打楽器に似た合成音（調波構造をもたない）。最大値を 1 にそろえて返す。
      kick  ：周波数が 120 Hz → 45 Hz に指数で下がる正弦波 × 速い減衰
      snare ：180 Hz と 330 Hz の短い音 ＋ 帯域を制限した雑音（響き線の音）
      hihat ：6 kHz より上の雑音 × とても速い減衰"""
    from scipy.signal import butter, sosfilt
    rng = np.random.default_rng(seed)
    n = int(round(sec * fs))
    t = np.arange(n) / fs
    if kind == "kick":
        f = 45.0 + 75.0 * np.exp(-t / 0.04)
        x = np.sin(2 * np.pi * np.cumsum(f) / fs) * np.exp(-t / 0.15)
    elif kind == "snare":
        tone = (np.sin(2 * np.pi * 180 * t) + 0.5 * np.sin(2 * np.pi * 330 * t)) * np.exp(-t / 0.05)
        sos = butter(2, [1000, min(7000, 0.45 * fs)], btype="bandpass", fs=fs, output="sos")
        noise = sosfilt(sos, rng.standard_normal(n)) * np.exp(-t / 0.08)
        x = 0.6 * tone + 2.0 * noise
    elif kind == "hihat":
        sos = butter(4, min(6000, 0.4 * fs), btype="highpass", fs=fs, output="sos")
        x = sosfilt(sos, rng.standard_normal(n)) * np.exp(-t / 0.025)
    else:
        raise ValueError('kind は "kick"・"snare"・"hihat"')
    return x / (np.max(np.abs(x)) + 1e-12)


# ---------------------------------------------------------------- ピッチ推定
def frame_signal(x, frame_length, hop_length, center=True):
    """(frame_length, n_frames) のフレームの並び。center=True なら前後に frame_length//2 個の 0 を足し、
    フレーム t の中心を時刻 t·hop_length に合わせる（librosa.yin・librosa.stft と同じ）"""
    x = np.asarray(x, dtype=float)
    if center:
        x = np.pad(x, (frame_length // 2, frame_length // 2))
    n_frames = 1 + (len(x) - frame_length) // hop_length
    idx = np.arange(frame_length)[:, None] + hop_length * np.arange(n_frames)[None, :]
    return x[idx]


def yin_difference(frame, W, tau_max):
    """差分関数 d(τ) = Σ_{j=0}^{W−1} (x_j − x_{j+τ})²、τ = 0, …, tau_max（論文の式 (6)）。
    積分の窓 W は τ によらず一定なので、frame は W + tau_max 点以上必要。
    展開すると d(τ) = r_0(0) + r_τ(0) − 2 r_0(τ)（式 (7)。r_t(τ) = Σ_j x_{t+j} x_{t+j+τ}）"""
    x = np.asarray(frame, dtype=float)
    if len(x) < W + tau_max:
        raise ValueError("frame が短い（W + tau_max 点以上にする）")
    e = np.concatenate([[0.0], np.cumsum(x ** 2)])          # e[i] = Σ_{j<i} x_j²
    d = np.empty(tau_max + 1)
    for tau in range(tau_max + 1):
        r = np.dot(x[:W], x[tau:tau + W])                    # r_0(τ)
        d[tau] = (e[W] - e[0]) + (e[tau + W] - e[tau]) - 2.0 * r
    return np.maximum(d, 0.0)                                # 丸め誤差で小さな負になるのを防ぐ


def cmndf(d):
    """累積平均正規化差分関数 d'(0) = 1、d'(τ) = d(τ) / [(1/τ) Σ_{j=1}^{τ} d(j)]（式 (8)）"""
    d = np.asarray(d, dtype=float)
    out = np.ones_like(d)
    cs = np.cumsum(d[1:])
    tau = np.arange(1, len(d))
    out[1:] = d[1:] * tau / (cs + 1e-300)
    return out


def _difference_librosa(frames, max_period):
    """librosa.yin の差分関数（フレーム全体の自己相関 r(k) と、先頭 k 点のエネルギーを使う）：
    d(k) = 2(r(0) − r(k)) − Σ_{m<k} x_m²。窓が縮む式 (2) の d に、フレームの末尾 k 点のエネルギーを
    足したものに当たる（論文の式 (6) とは少し違う）"""
    N = frames.shape[0]
    import scipy.fft
    n_pad = scipy.fft.next_fast_len(2 * N - 1, real=True)
    acf = scipy.fft.irfft(np.abs(scipy.fft.rfft(frames, n=n_pad, axis=0)) ** 2, n=n_pad, axis=0)[:max_period + 1]
    energy = np.cumsum(frames ** 2, axis=0)
    energy[0] = 0.0                                          # librosa は d(1) で x_0² を引かない（先に 0 を入れるため）
    d = np.zeros((max_period + 1, frames.shape[1]))
    k = np.arange(1, max_period + 1)
    d[1:] = 2.0 * (acf[0:1] - acf[k]) - energy[k - 1]
    return d


def yin(x, fs, fmin, fmax, frame_length=2048, hop_length=None, threshold=0.1, mode="librosa",
        center=True):
    """YIN（de Cheveigné & Kawahara 2002）の手順 2〜5。戻り値 (f0 [Hz], d' の値（非周期性の目安）)。
      手順2 差分関数 → 手順3 累積平均正規化 → 手順4 しきい値より深い最初の谷（無ければ全体の最小）
      → 手順5 放物線補間。周期の探索範囲は floor(fs/fmax) 〜 ceil(fs/fmin)。
    mode="librosa"：librosa.yin と同じ（差分は _difference_librosa、補間は d' で行う）
    mode="paper"  ：窓 W = frame_length − max_period を固定した式 (6)、補間の位置は d で求める
                    （論文の手順5：d' の補間は少し偏るので、位置は正規化前の d から取る）"""
    hop_length = hop_length or frame_length // 4
    min_p = int(np.floor(fs / fmax))
    max_p = min(int(np.ceil(fs / fmin)), frame_length - 1)
    frames = frame_signal(x, frame_length, hop_length, center)
    nT = frames.shape[1]
    if mode == "librosa":
        D = _difference_librosa(frames, max_p)
    elif mode == "paper":
        W = frame_length - max_p
        if W < max_p:
            raise ValueError("frame_length が短い（fmin の周期の 2 倍以上にする）")
        D = np.stack([yin_difference(frames[:, t], W, max_p) for t in range(nT)], axis=1)
    else:
        raise ValueError('mode は "librosa" か "paper"')
    # 累積平均正規化（τ = 1 からの平均で割る）
    cs = np.cumsum(D[1:], axis=0) / np.arange(1, max_p + 1)[:, None]
    Dn = D[min_p:max_p + 1] / (cs[min_p - 1:max_p] + 1e-300)     # τ = min_p, …, max_p
    nP = Dn.shape[0]
    # 谷（局所最小）：librosa.util.localmin と同じ決まり（左より小さく、右以下）
    trough = np.zeros_like(Dn, dtype=bool)
    trough[1:-1] = (Dn[1:-1] < Dn[:-2]) & (Dn[1:-1] <= Dn[2:])
    trough[0] = Dn[0] < Dn[1]
    trough[-1] = Dn[-1] < Dn[-2]
    below = trough & (Dn < threshold)
    idx = np.where(below.any(axis=0), np.argmax(below, axis=0), np.argmin(Dn, axis=0))
    # 放物線補間：3 点 (i−1, i, i+1) を通る放物線の頂点のずれ = −b/a、b = (v₊ − v₋)/2、a = v₊ + v₋ − 2v₀
    src = Dn if mode == "librosa" else D[min_p:max_p + 1]
    cols = np.arange(nT)
    i = np.clip(idx, 1, nP - 2)
    vm, v0, vp = src[i - 1, cols], src[i, cols], src[i + 1, cols]
    a = vp + vm - 2.0 * v0
    b = 0.5 * (vp - vm)
    ok = (idx > 0) & (idx < nP - 1) & (np.abs(b) < np.abs(a))
    shift = np.where(ok, -b / np.where(ok, a, 1.0), 0.0)
    period = min_p + idx + shift
    return fs / period, Dn[idx, cols]


def acf_pitch(x, fs, fmin, fmax, frame_length=2048, hop_length=None, center=True):
    """自己相関法（論文の手順1。比較用）：探索範囲で自己相関 r(τ) が最大の τ を周期とする。
    r(τ) は窓が縮む形（式 (2)）。放物線補間つき"""
    hop_length = hop_length or frame_length // 4
    min_p = int(np.floor(fs / fmax))
    max_p = min(int(np.ceil(fs / fmin)), frame_length - 1)
    frames = frame_signal(x, frame_length, hop_length, center)
    N = frames.shape[0]
    spec = np.fft.rfft(frames, n=2 * N, axis=0)
    r = np.fft.irfft(np.abs(spec) ** 2, n=2 * N, axis=0)[:max_p + 2]
    seg = r[min_p:max_p + 1]
    idx = np.argmax(seg, axis=0)
    cols = np.arange(frames.shape[1])
    i = np.clip(idx, 1, seg.shape[0] - 2)
    vm, v0, vp = seg[i - 1, cols], seg[i, cols], seg[i + 1, cols]
    a = vp + vm - 2.0 * v0
    b = 0.5 * (vp - vm)
    ok = (idx > 0) & (idx < seg.shape[0] - 1) & (np.abs(b) < np.abs(a))
    shift = np.where(ok, -b / np.where(ok, a, 1.0), 0.0)
    return fs / (min_p + idx + shift)


def gross_error(f_est, f_true, tol=0.2):
    """|f_est/f_true − 1| > tol なら True（YIN の論文の「グロスエラー」は tol = 0.2）。
    f_est が nan（無声と判定）のときも True（有声の区間だけを渡すこと）"""
    f_est = np.asarray(f_est, dtype=float)
    f_true = np.asarray(f_true, dtype=float)
    with np.errstate(invalid="ignore"):
        return ~(np.abs(f_est / f_true - 1.0) <= tol)


# ---------------------------------------------------------------- クロマ
def chroma_simple(X, fs, L, fmin=55.0, fmax=None):
    """STFT X (F, T) のパワーを、ビンの周波数に最も近い音名（12 個）ごとに足す。
    各フレームを最大 1 にそろえて (12, T) を返す（0 番が C）。fmin より低いビン・fmax より高いビンは使わない。
    librosa.feature.chroma_stft は三角形に近い重みと音の高さのずれの推定を使うので、値は一致しない"""
    P = np.abs(X) ** 2
    f = np.arange(P.shape[0]) * fs / L
    fmax = fmax or fs / 2
    use = (f >= fmin) & (f <= fmax)
    pc = np.round(hz_to_midi(f[use])).astype(int) % 12
    C = np.zeros((12, P.shape[1]))
    np.add.at(C, pc, P[use])
    return C / (C.max(axis=0, keepdims=True) + 1e-12)


In [ ]:
%%writefile common/nonlinear.py
"""第IV部の共通ライブラリ：非線形システムとダイオードクリッパ（IV-8 で定義）

  静的な非線形（記憶のない非線形）
    y = soft_clip(x, drive=1.0)          tanh(drive·x)/tanh(drive)（|x| ≤ 1 を |y| ≤ 1 に）
    y = hard_clip(x, c=0.5)              ±c で止める
  ダイオードクリッパ（直列抵抗 R → 出力にコンデンサ C と逆並列のダイオード 2 本。図IV-8-19）
    C dv/dt = (vin − v)/R − 2 Is sinh(v/(n VT))
    p = clipper_params(R=2.2e3, C=10e-9, Is=2.52e-9, n=1.752, VT=0.02585)
    v = clipper_static_newton(vin, p)    C = 0（記憶なし）のときの v（ニュートン法、ダイオード 2 本）
    v = clipper_static_lambertw(vin, p)  ダイオード 1 本で近似した陽な解（Lambert W。符号で向きを選ぶ）
    v = clipper_forward_euler(vin, fs, p)                陽解法（前進オイラー）
    v, it = clipper_backward_euler(vin, fs, p)           後退オイラー ＋ ニュートン法（it は反復回数）
    v, it = clipper_trapezoid(vin, fs, p)                台形則（双一次変換と同じ）＋ ニュートン法
    t, v = clipper_reference(f_vin, t_end, fs_out, p)    scipy の solve_ivp（Radau）で細かく解いた基準
    y = oversampled(x, func, factor)                     factor 倍に上げて func を通し、元に戻す
  ひずみの測り方
    h = harmonic_amps(x, fs, f0, n_harm)   最小二乗で k·f0 の振幅を求める（h[0] が基本波）
    r = thd(x, fs, f0, n_harm=20)          THD_F = √(Σ_{k≥2} A_k²)/A_1（基本波に対する比）
    r = thd_fft(x, fs, f0, n_harm=20)      同じものを FFT のビンで（整数周期を切り出す）
    db = inharmonic_db(x, fs, f0)          k·f0 以外の成分のパワーの割合 [dB]（折り返しの目安）
"""
import numpy as np

__all__ = ["soft_clip", "hard_clip", "clipper_params", "diode_i", "diode_g",
           "clipper_static_newton", "clipper_static_lambertw",
           "clipper_forward_euler", "clipper_backward_euler", "clipper_trapezoid",
           "clipper_reference", "oversampled", "harmonic_amps", "thd", "thd_fft", "inharmonic_db"]


# ---------------------------------------------------------------- 静的な非線形
def soft_clip(x, drive=1.0):
    """tanh 型のソフトクリップ。x = ±1 で ±1 になるように tanh(drive) で割る"""
    return np.tanh(drive * np.asarray(x, dtype=float)) / np.tanh(drive)


def hard_clip(x, c=0.5):
    """ハードクリップ：|x| > c を ±c で止める"""
    return np.clip(np.asarray(x, dtype=float), -c, c)


# ---------------------------------------------------------------- ダイオードクリッパ
def clipper_params(R=2.2e3, C=10e-9, Is=2.52e-9, n=1.752, VT=0.02585):
    """回路の定数（見本の値。Is・n は 1N4148 の SPICE モデルでよく使われる値で、データシートで要確認。
    第1章でダイオードを測っていれば、その推定値に置き換える）。nVT = n·VT [V]"""
    return dict(R=float(R), C=float(C), Is=float(Is), nVT=float(n * VT))


def diode_i(v, p):
    """逆並列のダイオード 2 本の電流 Is(e^{v/nVT} − 1) − Is(e^{−v/nVT} − 1) = 2 Is sinh(v/nVT)"""
    return 2.0 * p["Is"] * np.sinh(v / p["nVT"])


def diode_g(v, p):
    """その微分（小信号のコンダクタンス）2 Is cosh(v/nVT)/nVT"""
    return 2.0 * p["Is"] * np.cosh(v / p["nVT"]) / p["nVT"]


def _newton_scalar(fun, dfun, v0, tol=1e-12, max_iter=50, max_step=None):
    """1 変数のニュートン法。max_step を与えると、1 回に動く量をその大きさで止める（指数が大きすぎる点に
    飛んであふれるのを防ぐ。SPICE の「電圧の制限」と同じ考え方）。戻り値 (v, 反復回数)"""
    v = v0
    for it in range(1, max_iter + 1):
        step = fun(v) / dfun(v)
        if max_step is not None:
            step = np.clip(step, -max_step, max_step)
        v -= step
        if abs(step) < tol:
            return v, it
    raise RuntimeError(f"ニュートン法が {max_iter} 回で収束しない（v = {v}）")


def clipper_static_newton(vin, p, tol=1e-13):
    """C = 0 のとき (vin − v)/R = 2 Is sinh(v/nVT) を v について解く（各点ごとのニュートン法）"""
    vin = np.atleast_1d(np.asarray(vin, dtype=float))
    out = np.empty_like(vin)
    R = p["R"]
    for i, u in enumerate(vin):
        f = lambda v: v - u + R * diode_i(v, p)
        df = lambda v: 1.0 + R * diode_g(v, p)
        v0 = np.clip(u, -0.8, 0.8)                   # ダイオードの順電圧より大きくはならない
        out[i], _ = _newton_scalar(f, df, v0, tol=tol, max_step=4 * p["nVT"])
    return out


def clipper_static_lambertw(vin, p):
    """ダイオード 1 本（vin > 0 なら順方向の 1 本だけ）で近似した陽な解。
    (vin − v)/R = Is(e^{v/nVT} − 1) を解くと
        v = vin + R Is − nVT·W( (R Is/nVT)·exp((vin + R Is)/nVT) )
    W の中身はすぐ巨大になるので、W(e^z) = ω(z)（Wright の ω 関数、scipy.special.wrightomega）を使う：
        z = ln(R Is/nVT) + (vin + R Is)/nVT。vin < 0 は奇関数として −v(−vin)"""
    from scipy.special import wrightomega
    vin = np.asarray(vin, dtype=float)
    s = np.sign(vin)
    u = np.abs(vin)
    R, Is, V = p["R"], p["Is"], p["nVT"]
    z = np.log(R * Is / V) + (u + R * Is) / V
    v = u + R * Is - V * np.real(wrightomega(z))
    return s * v


def clipper_forward_euler(vin, fs, p, v0=0.0):
    """前進オイラー（陽解法）：v[n] = v[n−1] + h·f(v[n−1], vin[n−1])。h が大きいと発散する"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    v = np.empty(len(vin))
    vp = v0
    with np.errstate(over="ignore", invalid="ignore"):
        for n in range(len(vin)):
            u_prev = vin[n - 1] if n > 0 else 0.0
            vp = vp + h * ((u_prev - vp) / R - diode_i(vp, p)) / C
            v[n] = vp
    return v


def clipper_backward_euler(vin, fs, p, v0=0.0, tol=1e-10, max_iter=50):
    """後退オイラー（陰解法）：v[n] = v[n−1] + h·f(v[n], vin[n])。毎時刻、
        g(v) = v − v[n−1] − (h/C)·((vin[n] − v)/R − 2 Is sinh(v/nVT)) = 0
    をニュートン法で解く（g'(v) = 1 + (h/C)(1/R + 2 Is cosh(v/nVT)/nVT) > 0 なので解は1つ）。
    戻り値 (v, 各時刻のニュートン法の反復回数)"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    k = h / C
    v = np.empty(len(vin))
    its = np.empty(len(vin), dtype=int)
    vp = v0
    for n, u in enumerate(vin):
        g = lambda x: x - vp - k * ((u - x) / R - diode_i(x, p))
        dg = lambda x: 1.0 + k * (1.0 / R + diode_g(x, p))
        vp, its[n] = _newton_scalar(g, dg, vp, tol=tol, max_iter=max_iter, max_step=4 * p["nVT"])
        v[n] = vp
    return v, its


def clipper_trapezoid(vin, fs, p, v0=0.0, tol=1e-10, max_iter=50):
    """台形則：v[n] = v[n−1] + (h/2)(f(v[n−1], vin[n−1]) + f(v[n], vin[n]))（2 次の精度。
    線形の部分については IV-1 の双一次変換と同じ写像になる）。ニュートン法で解く"""
    h = 1.0 / fs
    R, C = p["R"], p["C"]
    k = h / (2.0 * C)
    v = np.empty(len(vin))
    its = np.empty(len(vin), dtype=int)
    vp, up = v0, 0.0
    for n, u in enumerate(vin):
        fp = (up - vp) / R - diode_i(vp, p)                  # 1 つ前の時刻の f（分かっている）
        g = lambda x: x - vp - k * (fp + (u - x) / R - diode_i(x, p))
        dg = lambda x: 1.0 + k * (1.0 / R + diode_g(x, p))
        vp, its[n] = _newton_scalar(g, dg, vp, tol=tol, max_iter=max_iter, max_step=4 * p["nVT"])
        v[n] = vp
        up = u
    return v, its


def clipper_reference(f_vin, t_end, fs_out, p, rtol=1e-10, atol=1e-13):
    """基準の解：scipy.integrate.solve_ivp（Radau：硬い方程式向けの陰的ルンゲ–クッタ法）で、
    入力を連続時間の関数 f_vin(t) として与え、時刻 n/fs_out の値を返す"""
    from scipy.integrate import solve_ivp
    R, C = p["R"], p["C"]
    rhs = lambda t, v: ((f_vin(t) - v) / R - diode_i(v, p)) / C
    jac = lambda t, v: np.array([[-(1.0 / R + diode_g(v[0], p)) / C]])
    t = np.arange(int(round(t_end * fs_out))) / fs_out
    sol = solve_ivp(rhs, (0.0, t[-1]), [0.0], method="Radau", t_eval=t, rtol=rtol, atol=atol, jac=jac,
                    max_step=0.25 / fs_out)
    if not sol.success:
        raise RuntimeError(sol.message)
    return t, sol.y[0]


def oversampled(x, func, factor=8):
    """x を factor 倍の標本化周波数に上げ（resample_poly）、func を通し、元の標本化周波数に戻す。
    上げるときの低域通過は「像」を消し、戻すときの低域通過は、非線形が作った高い倍音のうち
    元の fs/2 を超える分を、折り返す前に消す（IV-1-3-3）。func は (x_up, fs倍率) を受け取らず x_up だけを受け取る"""
    from scipy.signal import resample_poly
    if factor == 1:
        return func(np.asarray(x, dtype=float))
    xu = resample_poly(x, factor, 1)
    yu = func(xu)
    return resample_poly(yu, 1, factor)


# ---------------------------------------------------------------- ひずみの測り方
def harmonic_amps(x, fs, f0, n_harm=20):
    """x ≈ c + Σ_k (a_k cos 2πk f0 t + b_k sin 2πk f0 t) を最小二乗で当てはめ、
    振幅 A_k = √(a_k² + b_k²)（k = 1, …, n_harm。fs/2 以上の k は 0）を返す"""
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x)) / fs
    ks = [k for k in range(1, n_harm + 1) if k * f0 < fs / 2]
    cols = [np.ones_like(t)]
    for k in ks:
        cols += [np.cos(2 * np.pi * k * f0 * t), np.sin(2 * np.pi * k * f0 * t)]
    coef, *_ = np.linalg.lstsq(np.stack(cols, axis=1), x, rcond=None)
    A = np.zeros(n_harm)
    for i, k in enumerate(ks):
        A[k - 1] = np.hypot(coef[1 + 2 * i], coef[2 + 2 * i])
    return A


def thd(x, fs, f0, n_harm=20):
    """THD_F = √(A_2² + … + A_K²)/A_1（基本波に対する比。IEEE などの「F」の定義）。
    THD_R（全体の実効値に対する比）は THD_F/√(1 + THD_F²)"""
    A = harmonic_amps(x, fs, f0, n_harm)
    return float(np.sqrt(np.sum(A[1:] ** 2)) / A[0])


def thd_fft(x, fs, f0, n_harm=20):
    """FFT で THD_F を求める。f0 の周期がちょうど整数個入る長さ（fs/f0 が整数のとき）に切り、
    方形窓で FFT すると、k 倍音はビン k·(周期の数) に漏れなく入る（コヒーレント・サンプリング）"""
    P = fs / f0
    if abs(P - round(P)) > 1e-9:
        raise ValueError("fs/f0 が整数でない（整数周期を切り出せない）。thd() を使う")
    P = int(round(P))
    ncyc = len(x) // P
    seg = np.asarray(x[:ncyc * P], dtype=float)
    S = np.abs(np.fft.rfft(seg)) * 2 / len(seg)
    A = np.array([S[k * ncyc] if k * ncyc < len(S) - 1 else 0.0 for k in range(1, n_harm + 1)])
    return float(np.sqrt(np.sum(A[1:] ** 2)) / A[0])


def inharmonic_db(x, fs, f0, n_harm=None, width_hz=None):
    """k·f0（k = 0, 1, 2, …）の近く（±width_hz）以外のパワーの、全体に対する割合 [dB]。
    非線形が作る倍音のうち fs/2 を超えたものは、k·f0 でない周波数に折り返すので、これで折り返しを測る。
    窓は blackman-harris（漏れが小さい）、width_hz の既定は 6 ビン分"""
    from scipy.signal import get_window
    x = np.asarray(x, dtype=float)
    w = get_window("blackmanharris", len(x))
    S = np.abs(np.fft.rfft(x * w)) ** 2
    f = np.fft.rfftfreq(len(x), 1 / fs)
    width_hz = width_hz or 6 * fs / len(x)
    k = np.round(f / f0)
    near = np.abs(f - k * f0) <= width_hz
    return float(10 * np.log10(S[~near].sum() / S.sum()))


In [ ]:
%%writefile nmf8.py
"""IV-8：音楽の NMF の道具（第7章の nmf.py の KL 版を、初期値と「更新しない行列」を指定できる形にしたもの）

  W, H, cost = nmf_kl(X, W, H, n_iter=200, update_W=True, update_H=True, normalize=True)
      第7章と同じ乗法更新（KL、補助関数法）。W・H を与えた値から始める。
      乗法更新は 0 を 0 のまま保つので、W の 0 の場所を初期値で決めておけば、その形（調波構造など）が保たれる。
      update_W=False なら W を固定する（教師ありの NMF：学習済みの基底を使う）
  W0 = harmonic_templates(freqs, midis, n_harm=20, width_bins=1.0)
      候補の音（MIDI 番号）ごとに、k·f0（k = 1, …）の位置だけに山のある基底（ほかは 0）
  M = wiener_masks(W, H, groups, power=2)
      グループ（基底の番号の集合）ごとのマスク Σ_{k∈g}(w h)^p / Σ_all (w h)^p
  roll = frame_f(est, ref)  → (適合率, 再現率, F 値)（音×フレームの 0/1 の表どうし）
"""
import numpy as np

EPS = 1e-12


def cost_kl(X, Y):
    """一般化 KL ダイバージェンス（第7章と同じ）"""
    return float(np.sum(X * np.log((X + EPS) / (Y + EPS)) - X + Y))


def nmf_kl(X, W, H, n_iter=200, update_W=True, update_H=True, normalize=True):
    """第7章の nmf(X, k, "kl") と同じ順番の乗法更新（W → H → 列の和を 1 にそろえる）"""
    W = np.array(W, dtype=float)
    H = np.array(H, dtype=float)
    hist = []
    for _ in range(n_iter):
        if update_W:
            Y = W @ H + EPS
            W *= ((X / Y) @ H.T) / (H.sum(axis=1)[None, :] + EPS)
        if update_H:
            Y = W @ H + EPS
            H *= (W.T @ (X / Y)) / (W.sum(axis=0)[:, None] + EPS)
        if normalize and update_W:
            s = W.sum(axis=0) + EPS
            W /= s
            H *= s[:, None]
        hist.append(cost_kl(X, W @ H + EPS))
    return W, H, np.array(hist)


def harmonic_templates(freqs, midis, n_harm=20, width_bins=1.0, a4=440.0):
    """(F, len(midis)) の基底。音 m の列は、h·f0(m)（h = 1, …, n_harm、fs/2 未満）の位置を中心に
    幅 width_bins（ビン）のガウスの山を置き、中心から 3·width_bins より遠いビンは 0 にする。
    山の高さは 1/h（初期値。更新で変わる）。列の和を 1 にそろえる"""
    df = freqs[1] - freqs[0]
    W = np.zeros((len(freqs), len(midis)))
    for j, m in enumerate(midis):
        f0 = a4 * 2 ** ((m - 69) / 12)
        for h in range(1, n_harm + 1):
            if h * f0 >= freqs[-1]:
                break
            z = (freqs - h * f0) / (width_bins * df)
            W[:, j] += np.where(np.abs(z) <= 3, np.exp(-0.5 * z ** 2) / h, 0.0)
    return W / (W.sum(axis=0, keepdims=True) + EPS)


def wiener_masks(W, H, groups, power=2):
    """グループごとのマスク（第7章の nmf.py と同じ考え方。振幅の NMF なので 2 乗してパワーの比にする）"""
    parts = [(W[:, g] @ H[g, :]) ** power for g in groups]
    tot = sum(parts) + EPS
    return [p / tot for p in parts]


def frame_f(est, ref):
    """音×フレームの 0/1 の表どうしの、適合率・再現率・F 値（MIREX のフレーム単位の評価と同じ考え方）"""
    est = np.asarray(est, bool)
    ref = np.asarray(ref, bool)
    tp = np.sum(est & ref)
    p = tp / max(est.sum(), 1)
    r = tp / max(ref.sum(), 1)
    return p, r, (2 * p * r / (p + r) if p + r > 0 else 0.0)


In [ ]:
import sys, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="With fmin")
from common import music as mu, nonlinear as nl
from common.stft import stft, istft, bin_freqs
import nmf8


def listen(x, fs, rms=None):
    """最大 0.3 にそろえて（rms を与えると実効値をそろえて）再生ボタンを出す。normalize=False を消さない"""
    x = np.asarray(x, dtype=float)
    if rms is None:
        y = x / (np.max(np.abs(x)) + 1e-12) * 0.3
    else:
        y = np.clip(x / (np.sqrt(np.mean(x ** 2)) + 1e-12) * rms, -0.3, 0.3)
    display(Audio(y, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)

## 1. 楽音：倍音を足していく（教科書 IV-8-3-1）

In [ ]:
FS = 16000
cent = []
for K in [1, 3, 10, 36]:
    k = np.arange(1, K + 1)
    a = 1.0 / k
    cent.append(np.sum(k * 220 * a**2) / np.sum(a**2))
    print(f"倍音 {K:2d} 本：スペクトルの重心 {cent[-1]:6.1f} Hz")
    listen(mu.harmonic_tone(220.0, FS, 1.2, amps=a) * mu.adsr(int(1.2 * FS), FS, 0.02, 0.1, 0.8, 0.2), FS)
print("基本波の無い音（2〜10 倍音）")
listen(mu.harmonic_tone(220.0, FS, 1.2, amps=np.r_[0.0, 1.0 / np.arange(2, 11)]), FS)

In [ ]:
# 確かめ（教科書の手順 A）
ok(all(abs(c - t) < 0.06 for c, t in zip(cent, [220.0, 296.3, 415.8, 567.8])), f"重心が 1・3・10・36 本で 220.0・296.3・415.8・567.8 Hz（{np.round(cent, 1)}）")

## 2. ピッチ推定：自前の YIN と librosa.yin・pyin（教科書 IV-8-3-3・3-4）

In [ ]:
FL, HOP, FMIN, FMAX = 2048, 512, 60.0, 1100.0
rng = np.random.default_rng(0)
x = np.concatenate([mu.harmonic_tone(f, FS, 0.5, amps="saw", decay=0.3) for f in [110, 220.5, 330, 440, 196, 523.25]])
x = x + 0.01 * rng.standard_normal(len(x))
try:
    y_tr, _ = librosa.load(librosa.ex("trumpet"), sr=FS)   # 初回はダウンロードする
    y_tr = y_tr.astype(float)
    TRUMPET = True
except Exception as e:   # ダウンロードできないときは、合成のトランペット風の音（ノコギリ波に近い倍音＋ビブラート）に切り替える
    print("trumpet をダウンロードできなかったので、合成の音に切り替えます：", type(e).__name__)
    tt_ = np.arange(int(0.6 * FS)) / FS
    notes = []
    for f0_ in [392.0, 523.25, 466.16, 440.0, 349.23, 392.0]:
        ph = 2 * np.pi * np.cumsum(f0_ * (1 + 0.004 * np.sin(2 * np.pi * 5.5 * tt_))) / FS
        notes.append(sum(np.sin(k_ * ph) / k_ for k_ in range(1, 16)) * mu.adsr(len(tt_), FS, 0.03, 0.1, 0.8, 0.1))
    y_tr = np.concatenate(notes + [np.zeros(int(0.2 * FS))])
    y_tr = 0.5 * y_tr / np.max(np.abs(y_tr)) + 0.003 * rng.standard_normal(len(y_tr))
    TRUMPET = False
yin_diff = []
for name, sig in [("合成音", x), ("trumpet", y_tr)]:
    a, _ = mu.yin(sig, FS, FMIN, FMAX, FL, HOP)
    b = librosa.yin(sig, fmin=FMIN, fmax=FMAX, sr=FS, frame_length=FL, hop_length=HOP)
    yin_diff.append(np.max(np.abs(a - b)))
    print(f"{name}：自前（librosa モード）と librosa.yin の差の最大 {yin_diff[-1]:.1e} Hz")
print("偶数倍音が強い音（基本波の比 r）")
even = {}
for r_ in [0.5, 0.2, 0.1, 0.05]:
    amps = np.array([r_ if k == 1 else (1.0 / k if k % 2 == 0 else 0.02 / k) for k in range(1, 21)])
    e, _ = mu.yin(mu.harmonic_tone(220.0, FS, 1.0, amps=amps), FS, FMIN, FMAX, FL, HOP)
    even[r_] = np.median(e)
    print(f"  r = {r_:4.2f}：YIN {even[r_]:6.1f} Hz")
f_p, voiced, _ = librosa.pyin(y_tr, fmin=100.0, fmax=1100.0, sr=FS, frame_length=FL, hop_length=HOP)
f_y, _ = mu.yin(y_tr, FS, 100.0, 1100.0, FL, HOP)
both = voiced & np.isfinite(f_p)
dc = mu.cents(f_y[both], f_p[both])
print(f"trumpet：YIN と pYIN の |差| < 50 セント の割合 {np.mean(np.abs(dc) < 50):.3f}")
t = librosa.frames_to_time(np.arange(len(f_y)), sr=FS, hop_length=HOP)
plt.figure(figsize=(8, 3)); plt.semilogy(t, f_y, ".", ms=3, label="YIN（自前）"); plt.semilogy(t, f_p, "-", label="pYIN")
plt.xlabel("時間 [s]"); plt.ylabel("f0 [Hz]"); plt.legend(); plt.show()

In [ ]:
# 確かめ（教科書の手順 B）
ok(max(yin_diff) < 1e-10, f"自前の YIN と librosa.yin の差が 1e-13 Hz の桁（{max(yin_diff):.1e} Hz。許容 1e-10）")
ok(abs(even[0.5] - 220) < 2 and abs(even[0.2] - 220) < 2 and abs(even[0.1] - 440) < 5 and abs(even[0.05] - 440) < 5,
   f"偶数倍音が強い音で、r = 0.10 以下が 440 Hz 付近（オクターブ誤り）{ {k: round(v, 1) for k, v in even.items()} }")
if TRUMPET:
    ok(abs(np.mean(np.abs(dc) < 50) - 0.915) < 0.03, f"trumpet の YIN と pYIN の |差| < 50 セント の割合が 0.915 前後（{np.mean(np.abs(dc) < 50):.3f}）")
else:
    print("（合成の音なので、pYIN との一致の割合の確かめは省きます：", round(float(np.mean(np.abs(dc) < 50)), 3), "）")

## 3. 音楽の NMF：調波構造の制約つき（教科書 IV-8-3-5）

In [ ]:
L, S = 4096, 1024
MIDIS = np.arange(48, 85)
SCALE = [55, 57, 59, 60, 62, 64, 65, 67, 69, 71, 72, 74, 76, 77, 79]


def make_piece(seed, sec=8.0):
    rng = np.random.default_rng(seed)
    x = np.zeros(int(sec * FS)); score = []; t = 0.0
    while t < sec - 0.6:
        dur = float(rng.choice([0.25, 0.5, 0.75, 1.0]))
        for m in rng.choice(SCALE, size=int(rng.integers(1, 4)), replace=False):
            n0 = int(t * FS)
            tone = mu.harmonic_tone(float(mu.midi_to_hz(m)), FS, dur, amps="saw", decay=0.6, n_harm=15)
            tone *= np.minimum(1.0, (len(tone) - np.arange(len(tone))) / (0.03 * FS))
            x[n0:n0 + len(tone)] += tone[:len(x) - n0] * rng.uniform(0.5, 1.0)
            score.append((t, dur, int(m)))
        t += dur
    return x / np.max(np.abs(x)) * 0.5, score


def analyze(seed):
    x, score = make_piece(seed)
    Z = stft(x, L, S); X = np.abs(Z)
    W0 = nmf8.harmonic_templates(bin_freqs(L, FS), MIDIS)
    H0 = np.random.default_rng(0).uniform(0.1, 1.0, (len(MIDIS), X.shape[1])) * X.mean()
    W, H, _ = nmf8.nmf_kl(X, W0, H0, 200)
    R = np.zeros((len(MIDIS), X.shape[1]), bool); tt = np.arange(X.shape[1]) * S / FS
    for t0, d, m in score:
        R[m - MIDIS[0], (tt >= t0) & (tt < t0 + d)] = True
    return x, Z, W, H, R


dev = analyze(1)
grid = [0.05, 0.1, 0.15, 0.2, 0.3]
Hn = lambda W, H: H * W.sum(axis=0)[:, None]
f_dev = [nmf8.frame_f(Hn(dev[2], dev[3]) > th * Hn(dev[2], dev[3]).max(), dev[4])[2] for th in grid]
th = grid[int(np.argmax(f_dev))]
print("開発用の F 値：", np.round(f_dev, 3), "→ しきい値", th)
x, Z, W, H, R = analyze(2)
prf = nmf8.frame_f(Hn(W, H) > th * Hn(W, H).max(), R)
print("評価用の曲（種 2）の 適合率・再現率・F：", np.round(prf, 3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].imshow(R, origin="lower", aspect="auto", cmap="Greys"); ax[0].set_title("正解")
ax[1].imshow(np.log10(Hn(W, H) + 1e-3 * Hn(W, H).max()), origin="lower", aspect="auto", cmap="magma"); ax[1].set_title("H（log）")
plt.show()
listen(x, FS)
m = 60
Mk = nmf8.wiener_masks(W, H, [[m - 48], [j for j in range(len(MIDIS)) if j != m - 48]])[0]
listen(istft(Mk * Z, L, S, length=len(x)), FS)

In [ ]:
# 確かめ（教科書の手順 C。ここは評価用の曲が 1 曲だけなので、教科書の 5 曲の F 0.791 ± 0.029 より幅を広く取る）
ok(abs(th - 0.15) < 0.051, f"開発用の曲で選んだしきい値が 0.15 前後（{th}）")
ok(0.65 < prf[2] < 0.9, f"評価用の曲の F が 0.7〜0.8 前後（{prf[2]:.3f}。教科書の 5 曲の平均 0.791）")

## 4. HPSS：自前のメディアンフィルタと librosa（教科書 IV-8-3-6）

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view


def median_filter_axis(P, size, axis):
    P = np.moveaxis(P, axis, -1); h = size // 2
    pad = np.pad(P, [(0, 0)] * (P.ndim - 1) + [(h, size - 1 - h)], mode="symmetric")
    return np.moveaxis(np.median(sliding_window_view(pad, size, axis=-1), axis=-1), -1, axis)


def hpss_masks(A, kernel=31, power=2.0):
    Hm, Pm = median_filter_axis(A, kernel, 1), median_filter_axis(A, kernel, 0)
    Zm = np.maximum(Hm, Pm); bad = Zm < np.finfo(float).tiny; Zm = np.where(bad, 1.0, Zm)
    mh, mp = (Hm / Zm) ** power, (Pm / Zm) ** power
    MH = np.where(bad, 0.5, mh / (mh + mp + bad))
    return MH, 1.0 - MH


beat = 0.5; n = int(8 * FS); drums = np.zeros(n); harm = np.zeros(n)
for i in range(32):
    for kd in ["hihat"] + (["kick"] if i % 4 == 0 else []) + (["snare"] if i % 4 == 2 else []):
        s = mu.drum(kd, FS, 0.4, seed=i * 3 + len(kd)) * {"kick": 1.0, "snare": 0.8, "hihat": 0.25}[kd]
        n0 = int(i * beat / 2 * FS); drums[n0:n0 + len(s)] += s[:n - n0]
for b_, ch in enumerate([[48, 64, 67, 72], [45, 64, 69, 72], [41, 65, 69, 72], [43, 62, 67, 71]] * 2):
    n0 = int(b_ * 2.0 * FS)
    if n0 >= n:
        break
    for m in ch:
        harm[n0:n0 + int(2.0 * FS)] += mu.harmonic_tone(float(mu.midi_to_hz(m - (12 if m < 50 else 0))), FS, 2.0, amps="saw", n_harm=12)[:n - n0] * 0.25
mix = drums + harm; mix = mix / np.max(np.abs(mix)) * 0.5
Zm_ = stft(mix, 2048, 512); A = np.abs(Zm_)
MH, MP = hpss_masks(A)
MH_lib, MP_lib = librosa.decompose.hpss(A, kernel_size=31, power=2.0, mask=True)
print("自前と librosa.decompose.hpss のマスクの差の最大：", np.max(np.abs(MH - MH_lib)))
print("混合 → 調波 → 打撃")
listen(mix, FS); listen(istft(MH * Zm_, 2048, 512, length=len(mix)), FS); listen(istft(MP * Zm_, 2048, 512, length=len(mix)), FS)

In [ ]:
# 確かめ（教科書の手順 D）
ok(np.max(np.abs(MH - MH_lib)) < 1e-12 and np.max(np.abs(MP - MP_lib)) < 1e-12, f"自前と librosa.decompose.hpss のマスクの差が 0（{np.max(np.abs(MH - MH_lib)):.1e}。許容 1e-12）")

## 5. 被り音の抑圧：時間チャネル NMF（教科書 IV-8-3-7。短い版）

In [ ]:
import fast_bss_eval
EPS = 1e-12
src = np.stack([drums, harm, mu.harmonic_tone(523.25, FS, 8.0, amps="square", n_harm=10) * mu.adsr(n, FS, 0.05, 0.2, 0.6, 0.2),
                mu.harmonic_tone(110.0, FS, 8.0, amps="saw", n_harm=30, decay=2.0)])
src = np.stack([s / np.max(np.abs(s)) * 0.25 for s in src])
Ss = stft(src.T, 2048, 1024, "hamming")


def tcnmf(Xc, k, theta, alpha, vmax, prior=True, n_iter=200):
    X = np.abs(Xc).transpose(0, 2, 1) * (alpha / vmax); F, M, J = X.shape
    rng = np.random.default_rng(0)
    A = rng.uniform(0, 0.1, (F, M, M)); A[:, np.arange(M), np.arange(M)] = 1.0
    Sx = rng.uniform(0, 1, (F, M, J)); one = np.ones((F, M, J))
    for _ in range(n_iter):
        Y = A @ Sx + EPS
        if prior:
            A = ((k - 1) + A * ((X / Y) @ Sx.transpose(0, 2, 1))) / (1.0 / theta + one @ Sx.transpose(0, 2, 1))
        else:
            A = A * ((X / Y) @ Sx.transpose(0, 2, 1)) / (one @ Sx.transpose(0, 2, 1) + EPS)
        A[:, np.arange(M), np.arange(M)] = 1.0
        Y = A @ Sx + EPS
        Sx = Sx * (A.transpose(0, 2, 1) @ (X / Y)) / (A.transpose(0, 2, 1) @ one + EPS)
    return A, Sx


def wiener(Xc, A, Sx):
    P = Sx ** 2
    return (P / ((A ** 2) @ P + EPS)).transpose(0, 2, 1) * Xc


sdr = lambda ref, est: np.array([float(fast_bss_eval.sdr(r[None], e[None])[0]) for r, e in zip(ref, est)])
res = {"事前分布なし": [], "提案（論文の値）": [], "正解の A と |s|": []}
for seed in range(5, 10):
    rng = np.random.default_rng(seed)
    At = rng.uniform(0, 0.2, (Ss.shape[0], 4, 4)); At[:, np.arange(4), np.arange(4)] = 1.0
    Xc = np.einsum("fmn,ftn->ftm", At, Ss)
    xm = istft(Xc, 2048, 1024, "hamming", length=n).T
    base = sdr(src, xm)
    for name in res:
        if name == "正解の A と |s|":
            Y = wiener(Xc, At, np.abs(Ss).transpose(0, 2, 1))
        else:
            A, Sx = tcnmf(Xc, 1.25, 0.6, 0.006, np.max(np.abs(xm)), prior=(name != "事前分布なし"))
            Y = wiener(Xc, A, Sx)
        res[name].append(np.mean(sdr(src, istft(Y, 2048, 1024, "hamming", length=n).T) - base))
for name, v in res.items():
    print(f"{name:14s}：SDR の改善量 {np.mean(v):+.2f} ± {np.std(v, ddof=1):.2f} dB（混合 5 個）")
print("マイク 2（調波の楽器）→ 抑圧後")
listen(xm[1], FS); listen(istft(Y, 2048, 1024, "hamming", length=n).T[1], FS)

In [ ]:
# 確かめ（教科書の手順 E。教科書は 10 種・本番の長さで +0.12・+11.30・+12.21 dB。ここは混合 5 個の短い版なので幅を広く取る）
g0, g1, g2 = (np.mean(res[k]) for k in ["事前分布なし", "提案（論文の値）", "正解の A と |s|"])
ok(abs(g0) < 1, f"事前分布なしの改善量がほぼ 0 dB（{g0:+.2f} dB）")
ok(7 < g1 < g2 and 10 < g2 < 14, f"提案の改善量が大きく（{g1:+.2f} dB）、上限（正解の A と |s|、{g2:+.2f} dB）より小さい")

## 6. ひずみ：THD と相互変調（教科書 IV-8-3-9）

In [ ]:
FS48 = 48000
t = np.arange(FS48) / FS48
thd_rows = []
for A_ in [0.1, 0.5, 1.0]:
    xx = A_ * np.cos(2 * np.pi * 1000 * t)
    thd_rows.append((100 * (A_**2 / 12) / (1 - A_**2 / 4), 100 * nl.thd(xx - xx**3 / 3, FS48, 1000), 100 * nl.thd_fft(xx - xx**3 / 3, FS48, 1000)))
    print(f"A = {A_}：式 {thd_rows[-1][0]:.4f} %、最小二乗 {thd_rows[-1][1]:.4f} %、FFT {thd_rows[-1][2]:.4f} %")
for name, semis in [("完全5度", 7), ("長3度", 4)]:
    sig = sum(mu.harmonic_tone(110.0 * 2 ** (s_ / 12), FS, 2.0, amps="saw", n_harm=10, decay=1.0) for s_ in (0, semis))
    print(name, "（ひずませた音。実効値をそろえて再生）")
    listen(nl.oversampled(sig / np.max(np.abs(sig)), lambda u: np.tanh(8 * u), 8), FS, rms=0.03)

In [ ]:
# 確かめ（教科書の手順 F）
ok(all(abs(ls - f) < 1e-4 * f and abs(ff - f) < 1e-4 * f for f, ls, ff in thd_rows), f"THD の式・最小二乗・FFT が 4 桁一致（{np.round(thd_rows, 4).tolist()}）")

## 7. ダイオードクリッパ：静的な解と数値解法（教科書 IV-8-3-10・3-11）

In [ ]:
p = nl.clipper_params()
u = np.linspace(-3, 3, 301)
d_nw = 1e6 * np.max(np.abs(nl.clipper_static_newton(u, p) - nl.clipper_static_lambertw(u, p)))
print("ニュートン法（2 本）と Lambert W（1 本）の差の最大 [µV]：", d_nw)
err = {}
fin = lambda tt: np.sin(2 * np.pi * 1000 * tt)
for fs in [48000, 96000]:
    tt = np.arange(int(0.006 * fs)) / fs
    _, vr = nl.clipper_reference(fin, 0.006, fs, p)
    vb, _ = nl.clipper_backward_euler(fin(tt), fs, p); vt, _ = nl.clipper_trapezoid(fin(tt), fs, p)
    h = len(tt) // 2
    err[fs] = (1000 * np.sqrt(np.mean((vb[h:] - vr[h:])**2)), 1000 * np.sqrt(np.mean((vt[h:] - vr[h:])**2)))
    print(f"fs = {fs}：後退オイラー {err[fs][0]:.3f} mV、台形則 {err[fs][1]:.4f} mV")
vf = nl.clipper_forward_euler(fin(np.arange(480) / 48000), 48000, p)
diverged = not np.all(np.isfinite(vf)) or np.nanmax(np.abs(vf)) > 10
print("前進オイラー（48 kHz）：", "発散" if diverged else "有限")

In [ ]:
# 確かめ（教科書の手順 G）
ok(abs(d_nw - 5.42) < 0.05, f"ニュートン法（2 本のダイオード）と Lambert W（1 本で近似）の差の最大が 5.42 µV（{d_nw:.3f} µV。近似の差なので 0 にはならない）")
ok(abs(err[48000][0] - 4.36) < 0.1 and abs(err[48000][1] - 0.549) < 0.02, f"48 kHz の誤差が 後退オイラー 4.36 mV・台形則 0.549 mV（{err[48000][0]:.3f}・{err[48000][1]:.4f}）")
r_b, r_t = err[48000][0] / err[96000][0], err[48000][1] / err[96000][1]
ok(abs(r_b - 1.97) < 0.1 and abs(r_t - 4.00) < 0.2, f"fs を 2 倍にしたときの誤差の比が 1.97・4.00 前後（{r_b:.2f}・{r_t:.2f}。1 次と 2 次の精度）")
ok(diverged, "前進オイラーは 48 kHz で発散する")

## 8. 折り返しとオーバーサンプリング（教科書 IV-8-3-12）

In [ ]:
x1760 = np.sin(2 * np.pi * 1760 * np.arange(int(0.25 * FS48)) / FS48)
inh = {}
for fac in [1, 2, 8]:
    y = nl.oversampled(x1760, lambda v: nl.hard_clip(v, 0.3), fac)
    inh[fac] = nl.inharmonic_db(y[len(y)//5:-len(y)//5], FS48, 1760)
    print(f"倍率 {fac}：k·1760 Hz 以外の割合 {inh[fac]:.1f} dB")
from scipy.signal import chirp
tt = np.arange(4 * FS) / FS
sw = chirp(tt, 200, tt[-1], 4000, method="logarithmic")
print("ハードクリップ：倍率 1 → 倍率 8（とくに小さい音量で）")
listen(nl.oversampled(sw, lambda v: nl.hard_clip(v, 0.3), 1), FS, rms=0.03)
listen(nl.oversampled(sw, lambda v: nl.hard_clip(v, 0.3), 8), FS, rms=0.03)

In [ ]:
# 確かめ（教科書の手順 H）
ok(abs(inh[1] + 34.3) < 1 and abs(inh[8] + 60.0) < 1.5 and inh[1] > inh[2] > inh[8], f"ハードクリップの折り返しが 1 倍で −34.3 dB、8 倍で −60.0 dB（{ {k: round(v, 1) for k, v in inh.items()} }）")

## おわりに

- 自前の実装（YIN・HPSS・NMF・THD・クリッパ）が既存のライブラリ・式と一致することを確かめました。教科書の 7 節の確認問題も解いてみてください。
- 被り音の節は、音源分離の論文（Mizobuchi ほか、APSIPA ASC 2021、arXiv:2109.00237）の式を小さく作り直したものです。研究室のプログラムではありません。